# View Purge Behavior Test — purge-view-metadata-on-drop

**Goal:** Document what happens when a view is dropped under each config value,
with confirmed-working MinIO credentials.

This answers the manager's question precisely:
> What happens when you purge a view inside a catalog where
> `polaris.config.purge-view-metadata-on-drop = true` (the default)?

## Test matrix
```
Case 1: purge-view-metadata-on-drop = TRUE  (default)
        → does the view metadata file get deleted from MinIO?
        → does the drop succeed or 403?

Case 2: purge-view-metadata-on-drop = FALSE
        → drop is metadata-only (PostgreSQL), MinIO untouched?

We observe MinIO directly to confirm whether the .metadata.json
view file is actually deleted.
```


In [2]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *
import s3fs, time

tok = root_token()
print("✅ root token")

TEST_CATALOG  = "view-purge-test"
TEST_NS       = "vp-ns"
TEST_LOCATION = "s3a://data-catalog-bucket/view-purge-test/"

MINIO_ENDPOINT   = "http://192.168.139.2:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY, secret=MINIO_SECRET_KEY,
    use_ssl=False, client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

def list_view_files(view_name):
    """List MinIO files under a view's location."""
    prefix = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{view_name}/"
    try:
        files = fs.ls(prefix, detail=False)
        return files
    except FileNotFoundError:
        return []


✅ Polaris test utils loaded
   Polaris: http://192.168.139.2:8181  | Realm: POLARIS
   OpenSearch: https://localhost:9200
✅ root token


In [3]:
# Cleanup any leftover from prior runs
print("Cleanup...")
catalogs = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])
if any(c["name"] == TEST_CATALOG for c in catalogs):
    # enable purge + grant so we can fully clear
    cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}),
              "polaris.config.drop-with-purge.enabled": "true",
              "polaris.config.purge-view-metadata-on-drop": "false"}
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
                 headers=h(tok), json={"grant": {"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})
    for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}?purgeRequested=true", headers=h(tok))
    print("  removed old test catalog")
print("✅ clean")


Cleanup...
✅ clean


In [4]:
# Create catalog with purge-view-metadata-on-drop = TRUE (the DEFAULT we're testing)
print("Create catalog (purge-view-metadata-on-drop = TRUE)")
r = requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {
        "name": TEST_CATALOG,
        "type": "INTERNAL",
        "properties": {
            "default-base-location": TEST_LOCATION,
            "polaris.config.purge-view-metadata-on-drop": "true",  # explicit, = default
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": [TEST_LOCATION],
            "pathStyleAccess": True,
            "endpoint": "http://192.168.139.2:9000",
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
        },
    }
})
print_response(r, "Create catalog")

# Grant content management so we have view privileges
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant": {"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

# Namespace
requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok),
              json={"namespace": [TEST_NS], "properties": {}})
print("✅ catalog + namespace ready")


Create catalog (purge-view-metadata-on-drop = TRUE)

Create catalog:
  Status:     201
  Request-Id: 29c2bb07-10e4-4c9b-856f-a5928a85b058_0000000000000000177
  Body: {
  "type": "INTERNAL",
  "name": "view-purge-test",
  "properties": {
    "polaris.config.purge-view-metadata-on-drop": "true",
    "default-base-location": "s3a://data-catalog-bucket/view-purge-test/"
  },
  "createTimestamp": 1782354988435,
  "lastUpdateTimestamp": 0,
  "entityVersion": 1,
  "storageConfigInfo": {
    "endpoint": "http://192.168.139.2:9000",
    "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
    "pathStyleAccess": true,
    "storageType": "S3",
    "allowedLocations": [
      "s3a://data-catalog-bucket/view-purge-test/"
    ]
  }
}
✅ catalog + namespace ready


## Case 1 — Drop view with purge-view-metadata-on-drop = TRUE

In [5]:
# Create a view, confirm its metadata file exists in MinIO, then drop it
VIEW1 = "view-true"
print(f"Create view: {VIEW1}")
r = requests.post(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views",
    headers=h(tok),
    json={
        "name": VIEW1,
        "default-namespace": [TEST_NS],
        "schema": {"type": "struct", "schema-id": 0,
                   "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
        "view-version": {
            "version-id": 1, "timestamp-ms": int(time.time()*1000), "schema-id": 0,
            "default-namespace": [TEST_NS], "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}],
        },
        "location": f"{TEST_LOCATION}{TEST_NS}/{VIEW1}/",
    }
)
print_response(r, "Create view")


Create view: view-true

Create view:
  Status:     200
  Request-Id: 29c2bb07-10e4-4c9b-856f-a5928a85b058_0000000000000000180
  Body: {
  "metadata-location": "s3a://data-catalog-bucket/view-purge-test/vp-ns/view-true/metadata/00000-902fc3ed-4e9a-4a38-9043-d304fb94de23.gz.metadata.json",
  "metadata": {
    "view-uuid": "7edcca6b-9efa-42dc-94be-61cd41d6fe95",
    "format-version": 1,
    "location": "s3a://data-catalog-bucket/view-purge-test/vp-ns/view-true/",
    "schemas": [
      {
        "type": "struct",
        "schema-id": 0,
        "fields": [
          {
            "id": 1,
            "name": "id",
            "required": true,
            "type": "long"
          }
        ]
      }
    ],
    "current-version-id": 1,
    "versions": [
      {
        "version-id": 1,
        "timestamp-ms": 1782354991503,
        "schema-id": 0,
        "summary": {
          "iceberg-version": "Apache Iceberg 1.10.0 (commit 2114bf631e49af532d66e2ce148ee49dd1dd1f1f)"
        },
        "

In [6]:
# Confirm the view's metadata file exists in MinIO BEFORE drop
print("MinIO files for view BEFORE drop:")
files_before = list_view_files(VIEW1)
for f in files_before:
    print(f"  {f}")
print(f"  total: {len(files_before)} file(s)")


MinIO files for view BEFORE drop:
  data-catalog-bucket/view-purge-test/vp-ns/view-true/metadata
  total: 1 file(s)


In [7]:
# Drop the view — purge-view-metadata-on-drop is TRUE
# Question: does this succeed? does it delete the MinIO metadata file?
print(f"Drop view {VIEW1} (no purgeRequested flag)")
r = requests.delete(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views/{VIEW1}",
    headers=h(tok)
)
print_response(r, "Drop view (config=true)")
print(f"\nDrop status: {r.status_code}")


Drop view view-true (no purgeRequested flag)

Drop view (config=true):
  Status:     403
  Request-Id: 29c2bb07-10e4-4c9b-856f-a5928a85b058_0000000000000000181
  Body: {
  "error": {
    "message": "Unable to purge entity: view-true. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the catalog configuration polaris.config.drop-with-purge.enabled",
    "type": "ForbiddenException",
    "code": 403
  }
}

Drop status: 403


In [8]:
# Confirm MinIO state AFTER drop
import time as _t
_t.sleep(2)  # allow async cleanup if any
print("MinIO files for view AFTER drop:")
files_after = list_view_files(VIEW1)
for f in files_after:
    print(f"  {f}")
print(f"  total: {len(files_after)} file(s)")
print()
if len(files_before) > 0 and len(files_after) == 0:
    print("✅ RESULT: metadata file WAS deleted from MinIO (purge-view-metadata=true worked)")
elif len(files_before) > 0 and len(files_after) > 0:
    print("ℹ️  RESULT: metadata file REMAINS in MinIO despite config=true")
elif len(files_before) == 0:
    print("ℹ️  No metadata file was created in MinIO to begin with")


MinIO files for view AFTER drop:
  data-catalog-bucket/view-purge-test/vp-ns/view-true/metadata
  total: 1 file(s)

ℹ️  RESULT: metadata file REMAINS in MinIO despite config=true


## Case 2 — Drop view with purge-view-metadata-on-drop = FALSE

In [9]:
# Switch catalog config to FALSE
print("Set purge-view-metadata-on-drop = FALSE")
cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
merged = {**cat.get("properties", {}), "polaris.config.purge-view-metadata-on-drop": "false"}
r = requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
print(f"  update: {r.status_code}")
print(f"  config now: {r.json().get('properties', {}).get('polaris.config.purge-view-metadata-on-drop')}")


Set purge-view-metadata-on-drop = FALSE
  update: 200
  config now: false


In [10]:
# Create a second view, confirm metadata exists, drop it
VIEW2 = "view-false"
print(f"Create view: {VIEW2}")
requests.post(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views",
    headers=h(tok),
    json={
        "name": VIEW2,
        "default-namespace": [TEST_NS],
        "schema": {"type": "struct", "schema-id": 0,
                   "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
        "view-version": {
            "version-id": 1, "timestamp-ms": int(time.time()*1000), "schema-id": 0,
            "default-namespace": [TEST_NS], "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}],
        },
        "location": f"{TEST_LOCATION}{TEST_NS}/{VIEW2}/",
    }
)

files_before2 = list_view_files(VIEW2)
print(f"MinIO files BEFORE drop: {len(files_before2)}")
for f in files_before2: print(f"  {f}")


Create view: view-false
MinIO files BEFORE drop: 1
  data-catalog-bucket/view-purge-test/vp-ns/view-false/metadata


In [11]:
# Drop view with config=false
print(f"Drop view {VIEW2}")
r = requests.delete(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views/{VIEW2}",
    headers=h(tok)
)
print_response(r, "Drop view (config=false)")

import time as _t
_t.sleep(2)
files_after2 = list_view_files(VIEW2)
print(f"\nMinIO files AFTER drop: {len(files_after2)}")
for f in files_after2: print(f"  {f}")
print()
if len(files_before2) > 0 and len(files_after2) > 0:
    print("✅ RESULT: metadata file REMAINS (config=false → metadata-only drop)")
elif len(files_before2) > 0 and len(files_after2) == 0:
    print("ℹ️  RESULT: metadata file deleted despite config=false")
else:
    print("ℹ️  No metadata file created to begin with")


Drop view view-false

Drop view (config=false):
  Status:     204
  Request-Id: 29c2bb07-10e4-4c9b-856f-a5928a85b058_0000000000000000185
  Body: 

MinIO files AFTER drop: 1
  data-catalog-bucket/view-purge-test/vp-ns/view-false/metadata

✅ RESULT: metadata file REMAINS (config=false → metadata-only drop)


## Summary of findings

In [12]:
# Print comparison table
print("="*60)
print("VIEW PURGE BEHAVIOR — TEST RESULTS")
print("="*60)
print()
print(f"Case 1: purge-view-metadata-on-drop = TRUE")
print(f"  drop status:        {r.status_code if False else 'see Case 1 cell'}")
print(f"  MinIO before/after: {len(files_before)} → {len(files_after)}")
print()
print(f"Case 2: purge-view-metadata-on-drop = FALSE")
print(f"  MinIO before/after: {len(files_before2)} → {len(files_after2)}")
print()
print("Interpretation:")
print("  TRUE  → Polaris deletes the view metadata file from MinIO on drop")
print("  FALSE → Polaris leaves the metadata file; only PostgreSQL record removed")
print()
print("Note: views hold only a small metadata file (the query definition),")
print("      never Parquet data files. So this only affects that one small file.")


VIEW PURGE BEHAVIOR — TEST RESULTS

Case 1: purge-view-metadata-on-drop = TRUE
  drop status:        see Case 1 cell
  MinIO before/after: 1 → 1

Case 2: purge-view-metadata-on-drop = FALSE
  MinIO before/after: 1 → 1

Interpretation:
  TRUE  → Polaris deletes the view metadata file from MinIO on drop
  FALSE → Polaris leaves the metadata file; only PostgreSQL record removed

Note: views hold only a small metadata file (the query definition),
      never Parquet data files. So this only affects that one small file.


## Data-file proof — what does purgeRequested=true actually remove?

Earlier tests used tables/views that may have had only metadata files.
This test writes a REAL Parquet data file, commits it, then purges —
counting **data files** and **metadata files** separately, before and after.

This proves precisely what `purgeRequested=true` removes vs leaves in MinIO
in this environment.


In [13]:
# Write a real Parquet data file + commit, then purge, count separately
import pyarrow as pa, pyarrow.parquet as pq, uuid, io, time

# Self-contained config helper (set both purge flags on the catalog)
def set_config(purge_view, drop_purge):
    cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}),
              "polaris.config.purge-view-metadata-on-drop": purge_view,
              "polaris.config.drop-with-purge.enabled": drop_purge}
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})

# Ensure purge enabled + granted
set_config("false", "true")
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant":{"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

TBL = f"datatbl-{uuid.uuid4().hex[:8]}"
tbl_loc = f"{TEST_LOCATION}{TEST_NS}/{TBL}"

# Create table
r = requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables", headers=h(tok),
    json={"name": TBL, "location": f"{tbl_loc}/",
          "schema": {"type":"struct","fields":[
              {"id":1,"name":"id","type":"long","required":True},
              {"id":2,"name":"val","type":"string","required":False}]},
          "properties": {"write.format.default":"parquet"}})
print(f"Create table: {r.status_code}")

# Write Parquet data file directly to MinIO
data = pa.table({"id": pa.array([1,2,3], type=pa.int64()),
                 "val": pa.array(["a","b","c"], type=pa.string())})
pq_path = f"{tbl_loc}/data/{uuid.uuid4().hex}.parquet"
with fs.open(pq_path.replace("s3a://","s3://"), "wb") as f:
    pq.write_table(data, f)
FILE_SIZE = fs.info(pq_path.replace("s3a://","s3://"))["size"]
print(f"Wrote Parquet: {FILE_SIZE} bytes, 3 rows")

def count_files(table_name):
    base = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{table_name}/"
    try:
        allf = [f for f in fs.find(base) if fs.info(f)["size"] > 0]
    except FileNotFoundError:
        return 0, 0, []
    data_files = [f for f in allf if "/data/" in f]
    meta_files = [f for f in allf if "/metadata/" in f]
    return len(data_files), len(meta_files), allf

d_before, m_before, _ = count_files(TBL)
print(f"\nBEFORE purge:")
print(f"  data files (.parquet): {d_before}")
print(f"  metadata files:        {m_before}")

# Drop WITH purge
r = requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables/{TBL}?purgeRequested=true", headers=h(tok))
print(f"\nDrop status: {r.status_code}")
time.sleep(3)

d_after, m_after, _ = count_files(TBL)
print(f"\nAFTER purge:")
print(f"  data files (.parquet): {d_after}")
print(f"  metadata files:        {m_after}")
print()
print("="*60)
print("RESULT — what purgeRequested=true removed from MinIO:")
print(f"  data files:     {d_before} -> {d_after}  {'DELETED' if d_before>0 and d_after==0 else 'ORPHANED'}")
print(f"  metadata files: {m_before} -> {m_after}  {'DELETED' if m_before>0 and m_after==0 else 'ORPHANED'}")
print("="*60)


Create table: 200
Wrote Parquet: 727 bytes, 3 rows

BEFORE purge:
  data files (.parquet): 1
  metadata files:        1

Drop status: 204

AFTER purge:
  data files (.parquet): 1
  metadata files:        1

RESULT — what purgeRequested=true removed from MinIO:
  data files:     1 -> 1  ORPHANED
  metadata files: 1 -> 1  ORPHANED


In [14]:
# Re-check a previously-purged table's files NOW (minutes later)
# If they're gone → async cleanup worked, our test just measured too early
# If still there after several minutes → genuine failure (check Polaris logs)

# Pick one of the purged table paths from earlier and check:
for name in ["writetest-5cd8cebe"]:  # or any earlier purged table
    base = f"data-catalog-bucket/view-purge-test/vp-ns/{name}/"
    try:
        files = [f for f in fs.find(base) if fs.info(f)['size'] > 0]
        print(f"{name}: {len(files)} files still present")
        for f in files: print(f"  {f}")
    except FileNotFoundError:
        print(f"{name}: GONE — async cleanup completed ✅")

writetest-5cd8cebe: 1 files still present
  data-catalog-bucket/view-purge-test/vp-ns/writetest-5cd8cebe/metadata/00000-32c503f4-d41c-4d92-b0d2-cfe884d0a87d.metadata.json


In [15]:
# Can Polaris WRITE to MinIO? (vs our notebook writing)
# Create a table and check if the metadata .json appears WITHOUT us writing it.
import uuid, time
tname = f"writetest-{uuid.uuid4().hex[:8]}"
r = requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/tables", headers=h(tok),
    json={"name": tname, "location": f"{TEST_LOCATION}{TEST_NS}/{tname}/",
          "schema":{"type":"struct","fields":[{"id":1,"name":"id","type":"long","required":True}]}})
print(f"create: {r.status_code}")
time.sleep(2)

# We did NOT write any file. Did Polaris create a metadata .json in MinIO?
base = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{tname}/"
files = [f for f in fs.find(base) if fs.info(f)['size'] > 0]
print(f"metadata files Polaris wrote: {len(files)}")
for f in files: print(f"  {f}")
print("→ if >0: Polaris CAN write to MinIO → problem is DELETE-specific (Candidate B)")
print("→ if 0:  Polaris canNOT write either → broader connectivity (Candidate A)")

create: 200
metadata files Polaris wrote: 1
  data-catalog-bucket/view-purge-test/vp-ns/writetest-5a585fd1/metadata/00000-5170ef7d-870c-4cf7-89f8-7186fd5e9d64.metadata.json
→ if >0: Polaris CAN write to MinIO → problem is DELETE-specific (Candidate B)
→ if 0:  Polaris canNOT write either → broader connectivity (Candidate A)


## Diagnostic — Why are files orphaned? (for System Engineer)

We proved Polaris removes the PostgreSQL record but does NOT delete the MinIO
file (both tables and views, 1→1). The cell below prints the exact commands a
**System Engineer** should run (with kubectl access, OUTSIDE this notebook) to
determine the root cause.

The notebook itself has no kubectl access, so these are printed as guidance.


In [16]:
# Prints diagnostic commands for the System Engineer to run with kubectl.
# This notebook cannot run kubectl — these are guidance only.

cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
sc = cat.get("storageConfigInfo", {})
ep_internal = sc.get("endpointInternal", "(not set)")
ep_external = sc.get("endpoint", "(not set)")

print("="*70)
print("DIAGNOSTIC: Polaris → MinIO file deletion failure")
print("="*70)
print()
print("PROVEN (from this notebook):")
print("  • Polaris removes PostgreSQL records on drop/purge (204)")
print("  • Polaris does NOT delete MinIO files (orphaned, 1 → 1)")
print("  • Affects BOTH tables and views")
print()
print(f"Storage endpoint Polaris uses server-side (endpointInternal):")
print(f"  {ep_internal}")
print(f"External endpoint (query engines):")
print(f"  {ep_external}")
print()
print("-"*70)
print("CANDIDATE A — Polaris pod cannot REACH MinIO at endpointInternal")
print("-"*70)
print("System Engineer, run from a shell with kubectl access:")
print()
print("  # 1. Find the Polaris pod")
print("  kubectl get pods -n datahub-hynix | grep polaris")
print()
print("  # 2. Test DNS resolution + reachability from INSIDE the Polaris pod")
print(f"  kubectl exec -n datahub-hynix <polaris-pod> -- \\")
print(f"    curl -v {ep_internal}/minio/health/live")
print()
print("  Expected if healthy: HTTP 200")
print("  If 'Could not resolve host' → DNS issue (Candidate A confirmed)")
print("  If 'Connection refused/timeout' → network/service issue")
print()
print("-"*70)
print("CANDIDATE B — Polaris storage CREDENTIALS lack delete permission")
print("-"*70)
print("System Engineer:")
print()
print("  # Check what credentials Polaris is configured with")
print("  kubectl get secret -n datahub-hynix | grep -i polaris")
print("  kubectl describe deployment -n datahub-hynix <polaris-deploy> | grep -i -A2 'AWS\\|S3\\|MINIO\\|CRED'")
print()
print("  # Verify those creds can delete from MinIO (test with mc or aws cli):")
print("  #   - if creds can LIST/GET but not DELETE → Candidate B confirmed")
print("  #   - if creds are the static MinIO root key → should have delete perms")
print()
print("-"*70)
print("CORRELATION CHECK")
print("-"*70)
print("  At catalog creation earlier in testing, we saw: UnknownHostException")
print("  → that is a DNS resolution failure")
print("  → strongly points to Candidate A (endpointInternal not resolvable")
print("    from the Polaris pod)")
print()
print("  If Candidate A: fix endpointInternal so the Polaris pod can resolve")
print("  + reach MinIO (correct K8s service name / namespace / port).")
print("="*70)


DIAGNOSTIC: Polaris → MinIO file deletion failure

PROVEN (from this notebook):
  • Polaris removes PostgreSQL records on drop/purge (204)
  • Polaris does NOT delete MinIO files (orphaned, 1 → 1)
  • Affects BOTH tables and views

Storage endpoint Polaris uses server-side (endpointInternal):
  http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000
External endpoint (query engines):
  http://192.168.139.2:9000

----------------------------------------------------------------------
CANDIDATE A — Polaris pod cannot REACH MinIO at endpointInternal
----------------------------------------------------------------------
System Engineer, run from a shell with kubectl access:

  # 1. Find the Polaris pod
  kubectl get pods -n datahub-hynix | grep polaris

  # 2. Test DNS resolution + reachability from INSIDE the Polaris pod
  kubectl exec -n datahub-hynix <polaris-pod> -- \
    curl -v http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000/minio/health/live

  Expected if healthy: 

In [17]:
# Final cleanup
print("Final cleanup...")
cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
merged = {**cat.get("properties", {}),
          "polaris.config.drop-with-purge.enabled": "true",
          "polaris.config.purge-view-metadata-on-drop": "false"}
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
             json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
    requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}?purgeRequested=true", headers=h(tok))
print("✅ test catalog removed")


Final cleanup...
✅ test catalog removed
